In [1]:
import torch
import pandas as pd
import numpy as np
from transformers import DistilBertTokenizer, DistilBertModel
import re
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from torch.optim import Adam
from torch import nn

In [2]:
device='cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/emotions-dataset")

print("Path to dataset files:", path)

100%|██████████| 14.5M/14.5M [00:00<00:00, 118MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/bhavikjikadara/emotions-dataset/versions/1


In [5]:
data=pd.read_csv(f"{path}/emotions.csv")

In [6]:
data.columns=['content','sentiment']

In [7]:
def convert_lower(text):
    text=re.sub(r"[=\[]", "", text)
    text=re.sub(r"\s+", " ", text).strip()
    return text.lower()

data['content']=data['content'].apply(convert_lower)


In [8]:
label_encoder=LabelEncoder()
data['sentiment']=label_encoder.fit_transform(data['sentiment'])

In [9]:
data['sentiment'].value_counts()

,count
sentiment,
1,141067
0,121187
3,57317
4,47712
2,34554
5,14972


In [10]:
X=data['content']
y=data['sentiment']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [11]:
tokenizer=DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [12]:
class CustomDataset(Dataset):
  def __init__(self,X,y,tokenizer):
    self.X=X
    self.y=y
    self.tokenizer=tokenizer
  def __len__(self):
    return len(self.X)
  def __getitem__(self,idx):
    text=self.X.iloc[idx]
    encoding=self.tokenizer(text,return_tensors="pt",max_length=128,truncation=True,padding="max_length")
    input_ids=encoding['input_ids']
    attention_mask=encoding['attention_mask']
    label=self.y.iloc[idx]
    return {'input_ids':input_ids.squeeze(0),'attention_mask':attention_mask.squeeze(0),'label':torch.tensor(label,dtype=torch.long)}

In [13]:
train_dataset=CustomDataset(X_train,y_train,tokenizer)
test_dataset=CustomDataset(X_test,y_test,tokenizer)

In [14]:
train_dataloader=DataLoader(train_dataset,batch_size=32,shuffle=True)
test_dataloader=DataLoader(test_dataset,batch_size=32,shuffle=False)

In [15]:
embedding=DistilBertModel.from_pretrained('distilbert-base-uncased')

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [16]:
class Classifier(nn.Module):
  def __init__(self,embedding):
    super().__init__()
    self.embedding=embedding
    for param in self.embedding.parameters():
            param.requires_grad = False
    self.classifier=nn.Sequential(
        nn.Linear(768,256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(256,6)
    )
  def forward(self,input_ids,attention_mask):
    output=self.embedding(input_ids=input_ids,attention_mask=attention_mask)
    output=output.last_hidden_state[:,0,:]
    output=self.classifier(output)
    return output

In [17]:
model=Classifier(embedding)
model=model.to(device)

In [18]:
loss=torch.nn.CrossEntropyLoss()
loss=loss.to(device)
optimizer=Adam(model.parameters(),lr=1e-5)
epochs=5
for param in model.embedding.transformer.layer[-2:].parameters():
    param.requires_grad = True

In [19]:
for epoch in range(epochs):
  total_loss=0
  count=0
  for batch in train_dataloader:
    count+=1
    input_ids=batch['input_ids']
    input_ids=input_ids.to(device)
    attention_mask=batch['attention_mask']
    attention_mask=attention_mask.to(device)
    labels=batch['label']
    labels=labels.to(device)

    optimizer.zero_grad()
    output=model(input_ids,attention_mask)
    loss_value=loss(output,labels)
    total_loss=total_loss+loss_value.item()
    loss_value.backward()
    optimizer.step()
    print(f'Epoch {epoch+1} Batch: {count} Batch Loss: {loss_value.item()}')
  print(f'Epoch {epoch+1}: Loss {total_loss/len(train_dataloader)}')

Streaming output truncated to the last 5000 lines.
Epoch 5 Batch: 5423 Batch Loss: 0.14627890288829803
Epoch 5 Batch: 5424 Batch Loss: 0.018289512023329735
Epoch 5 Batch: 5425 Batch Loss: 0.09417078644037247
Epoch 5 Batch: 5426 Batch Loss: 0.13043703138828278
Epoch 5 Batch: 5427 Batch Loss: 0.06975381076335907
Epoch 5 Batch: 5428 Batch Loss: 0.11338496208190918
Epoch 5 Batch: 5429 Batch Loss: 0.08870363235473633
Epoch 5 Batch: 5430 Batch Loss: 0.12606924772262573
Epoch 5 Batch: 5431 Batch Loss: 0.04742039740085602
Epoch 5 Batch: 5432 Batch Loss: 0.025557631626725197
Epoch 5 Batch: 5433 Batch Loss: 0.04434718191623688
Epoch 5 Batch: 5434 Batch Loss: 0.04872887581586838
Epoch 5 Batch: 5435 Batch Loss: 0.18269884586334229
Epoch 5 Batch: 5436 Batch Loss: 0.1990222930908203
Epoch 5 Batch: 5437 Batch Loss: 0.047504037618637085
Epoch 5 Batch: 5438 Batch Loss: 0.07348461449146271
Epoch 5 Batch: 5439 Batch Loss: 0.07413759082555771
Epoch 5 Batch: 5440 Batch Loss: 0.06744228303432465
Epoch 5 Bat

In [20]:
import torch

def evaluate_accuracy(model, test_dataloader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in test_dataloader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)

            outputs = model(input_ids, attention_mask)
            _, predicted = torch.max(outputs, dim=1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = correct / total
    return accuracy
# train_accuracy = evaluate_accuracy(model, train_dataloader, device)
# print(f"Train Accuracy: {train_accuracy:.4f}")
test_accuracy = evaluate_accuracy(model, test_dataloader, device)
print(f"Test Accuracy: {test_accuracy:.4f}")



Test Accuracy: 0.9343


In [21]:
# model.eval()
# all_top2_correct = []

# with torch.no_grad():
#     for batch in test_dataloader:  # Use test_dataloader for evaluation
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         labels = batch['label'].to(device)

#         out = model(input_ids, attention_mask)

#         # Get top-2 predicted class indices
#         top2_preds = torch.topk(out, k=2, dim=1).indices

#         # Check if the true label is in the top-2 predictions
#         correct = top2_preds.eq(labels.view(-1, 1)).any(dim=1)

#         all_top2_correct.extend(correct.cpu().numpy())

# top2_acc = np.mean(all_top2_correct)
# print(f"Test Top-2 Accuracy: {top2_acc:.2f}")


In [22]:
# model.eval()
# all_top2_correct = []

# with torch.no_grad():
#     for batch in train_dataloader:
#         input_ids = batch['input_ids'].to(device)
#         attention_mask = batch['attention_mask'].to(device)
#         labels = batch['label'].to(device)
#         out = model(input_ids, attention_mask)
#         top2_preds = torch.topk(out, k=2, dim=1).indices
#         correct = top2_preds.eq(labels.view(-1, 1)).any(dim=1)

#         all_top2_correct.extend(correct.cpu().numpy())

# top2_acc = np.mean(all_top2_correct)
# print(f"Train Top-2 Accuracy: {top2_acc:.2f}")
